# Code

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torch.nn as nn
import torch.optim as optim

# Define data transformations for grayscale images
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.Resize((48, 48)),  # Resize to 48x48
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalize for 1 channel
])

# Load the dataset from the folder structure
# ImageFolder expects structure: train/class1/, train/class2/, etc.
train_dataset = ImageFolder(root='train', transform=transform)

# Create data loader
train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=4
)

# Print dataset information
print(f"Number of classes: {len(train_dataset.classes)}")
print(f"Classes: {train_dataset.classes}")
print(f"Number of images: {len(train_dataset)}")
print(f"Class to index mapping: {train_dataset.class_to_idx}")

# Load ResNet18 and modify for grayscale input and 7 classes
resnet18_model = models.resnet18(pretrained=True)

# Modify first layer for grayscale (1 channel)
pretrained_weight = resnet18_model.conv1.weight.data
averaged_weight = pretrained_weight.mean(dim=1, keepdim=True)
resnet18_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet18_model.conv1.weight.data = averaged_weight

# Modify final layer for 7 classes
num_classes = 7
resnet18_model.fc = nn.Linear(resnet18_model.fc.in_features, num_classes)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet18_model = resnet18_model.to(device)


# Start training
# train_model(resnet18_model, train_loader, criterion, optimizer, num_epochs=10)

# Save the model
# torch.save(resnet18_model.state_dict(), 'emotion_resnet18.pth')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet18_model.parameters()), lr=0.0001)

# Training loop
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            # Zero the gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Statistics
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if (i + 1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], '
                      f'Loss: {loss.item():.4f}')
        
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}] completed: '
              f'Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')
    
    print('Training completed!')
    torch.save(model.state_dict(), 'emotion_resnet18.pth')
    print('Model saved as emotion_resnet18.pth')

In [ ]:
train_model(resnet18_model,train_loader, criterion, optimizer, 1)